# Voice2Reel 10 — AI Story Director (One Colab Notebook)
غيّر **IDEA** ثم شغّل خلايا **1، 3، 4، 5، 6** لإنشاء فيديو جديد. الخلية 2 للتدريب فقط عند الحاجة.

**السيناريو يُكتب بواسطة Qwen + Adapter مُدرّب على صيغة Scene DSL.** إذا لم يتوفر Adapter صالح، تتوقف العملية برسالة واضحة بدل توليد مشاهد ثابتة مضللة. الموديل يستطيع اختيار العناصر من مكتبة محدودة (راوتر، كابل، Wi-Fi، هاتف، لابتوب، سيرفر، سحابة)؛ الأفكار الأخرى قد تحصل على مشاهد بلا عناصر مناسبة. لا يوجد بحث صور تلقائي أو تحريك فيزيائي متقدم حتى الآن.

التدريب يحتاج GPU؛ توليد الصوت يحتاج إنترنت؛ النتائج تحفظ في Drive.

## 1 — إعدادات الفكرة والشخصية

In [ ]:
#@title 🎬 إعدادات الفيديو
IDEA = "اشرح كيف يعمل الراوتر وكيف يتصل بالإنترنت ويوزع الواي فاي" #@param {type:"string"}
LANGUAGE = "ar" #@param ["ar","en"]
CHARACTER = "classic" #@param ["classic","bold","neon","robot","sketch","ribbon","minimal","professor","astro","shadow","pixel","ink"]
RUN_TTS = True #@param {type:"boolean"}
RUN_RENDER = True #@param {type:"boolean"}
RUN_TRAINING = False #@param {type:"boolean"}
DATASET_COUNT = 1000 #@param {type:"integer"}
EPOCHS = 3 #@param {type:"integer"}
PATIENCE = 2 #@param {type:"integer"}
ADAPTER_OVERRIDE = "" #@param {type:"string"}
assert IDEA.strip() and len(IDEA) <= 2000
assert 1 <= DATASET_COUNT <= 2000000
print("💡", IDEA)

## 2 — تثبيت البيئة وربط Drive (مرة واحدة لكل جلسة)

In [ ]:
#@title Setup
import os, sys, subprocess, hashlib
from pathlib import Path
from IPython.display import display
from google.colab import drive
drive.mount("/content/drive")
REPO = Path("/content/Voice2Reel")
STORE = Path("/content/drive/MyDrive/Voice2Reel_09")
STORE.mkdir(parents=True, exist_ok=True)
if not (REPO / ".git").exists():
    subprocess.run(["git","clone","--branch","test-new-data",
                    "https://github.com/garaahmad/Voice2Reel.git",str(REPO)],check=True)
else:
    subprocess.run(["git","pull","--ff-only","origin","test-new-data"],cwd=str(REPO),check=True)
def run(*args, cwd=REPO, timeout=None):
    cmd=list(map(str,args))
    print("▶", " ".join(cmd), flush=True)
    subprocess.run(cmd,cwd=str(cwd),check=True,timeout=timeout)
run(sys.executable,"-m","pip","install","-q","-r","requirements/character-lab.txt")
run(sys.executable,"-m","pip","install","-q","-e",".","edge-tts==7.3.1")
sys.path.insert(0,str(REPO))
from voice2reel.character_templates import render_preview,lock_template,load_lock
from voice2reel.production_contract import ProductionPlan
# Each idea gets an isolated directory; changing the idea cannot reuse stale audio.
job_id = hashlib.sha256((IDEA+"|"+CHARACTER+"|"+LANGUAGE).encode()).hexdigest()[:12]
RESULTS = STORE/"ideas"/job_id
RESULTS.mkdir(parents=True,exist_ok=True)
DATA = STORE/"dataset"
MODEL_DIR = STORE/"qwen_adapter_run"
LOCK = RESULTS/"character_lock.json"
PLAN_PATH = RESULTS/"scene.json"
VOICE = RESULTS/"voice.mp3"
VIDEO = RESULTS/"final.mp4"
MUSIC = RESULTS/"music.mp3"
lock_template(CHARACTER, LOCK)
display(render_preview(CHARACTER))
print("📁", RESULTS)

## 3 — إعداد بيانات التدريب وتدريب Qwen (اختياري، عند الحاجة فقط)
إذا لديك Adapter مدرب حديثًا، اترك RUN_TRAINING=False. توليد بيانات جديدة وحده لا يدرّب الموديل. لا تستخدم Adapter قديمًا وتتوقع منه معرفة العناصر الجديدة.

In [ ]:
#@title Optional Qwen training
import torch
if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable T4 GPU first")
    run(sys.executable,"-m","pip","install","-q","-r","requirements/text.txt")
    # Torchvision can conflict with text-only Transformers on Colab.
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchvision"],check=True)
    run(sys.executable,"-m","scripts.build_large_scene_curriculum",
        "--count",DATASET_COUNT,"--output",DATA,"--check-every",max(1,min(DATASET_COUNT,1000)))
    idx=1
    while (STORE/f"qwen_adapter_idea_{idx}").exists():
        idx+=1
    train_output=STORE/f"qwen_adapter_idea_{idx}"
    run(sys.executable,"-u","-m","voice2reel.train_python_scene",
        "--train",DATA/"train.jsonl","--validation",DATA/"validation.jsonl",
        "--output",train_output,"--epochs",EPOCHS,"--patience",PATIENCE,timeout=21600)
    ADAPTER_OVERRIDE=str(train_output/"adapter")
    print("✅ Adapter:",ADAPTER_OVERRIDE)
else:
    print("⏭ Training skipped; using existing adapter.")

## 4 — توليد السيناريو بواسطة Qwen من IDEA
**مهم:** لا يوجد fallback ثابت. عند فشل النموذج تظهر الرسالة الحقيقية. سيُحفظ نص DSL بجانب JSON لمراجعته.

In [ ]:
#@title 🧠 Qwen writes a new story
from pathlib import Path
import subprocess, json
candidates=[]
if ADAPTER_OVERRIDE.strip():
    candidates.append(Path(ADAPTER_OVERRIDE.strip()))
candidates.extend(sorted(STORE.glob("qwen_adapter_idea_*/adapter"),reverse=True))
candidates.extend(sorted(STORE.glob("qwen_adapter_run*/adapter"),reverse=True))
ADAPTER=next((p for p in candidates if (p/"adapter_config.json").is_file()),None)
if ADAPTER is None:
    raise FileNotFoundError(
        "No trained Qwen adapter found. Enable RUN_TRAINING in cell 1 and run cell 3, "
        "or set ADAPTER_OVERRIDE to a folder containing adapter_config.json.")
print("🧠 Adapter:",ADAPTER)
run(sys.executable,"-m","pip","install","-q","-r","requirements/text.txt")
subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchvision"],check=True)
cmd=[sys.executable,"-u","-m","voice2reel.generate_python_scene",
     "--idea",IDEA,"--lock",str(LOCK),"--adapter",str(ADAPTER),"--output",str(PLAN_PATH)]
proc=subprocess.run(cmd,cwd=str(REPO),text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(proc.stdout,flush=True)
if proc.returncode:
    raise RuntimeError("Qwen generation failed; see traceback above.")
plan=ProductionPlan.model_validate_json(PLAN_PATH.read_text(encoding="utf-8"))
plan=plan.model_copy(update={"width":1080,"height":1920})
PLAN_PATH.write_text(plan.model_dump_json(indent=2),encoding="utf-8")
print("🎞️ Storyboard:",len(plan.shots),"shots")
for i,shot in enumerate(plan.shots,1):
    print(i,shot.seconds,"sec |",shot.narration,"| objects:",[o.kind for o in shot.objects])
print("📄",PLAN_PATH)

## 5 — توليد الصوت من السيناريو الجديد
لا تستخدم صوت فكرة قديمة. تتطلب خدمة Edge TTS اتصال إنترنت.

In [ ]:
#@title 🔊 Voiceover
if RUN_TTS:
    if VOICE.exists(): VOICE.unlink()
    run(sys.executable,"-m","voice2reel.production_voice",
        "--plan",PLAN_PATH,"--output",VOICE,"--language",LANGUAGE,timeout=600)
    if not VOICE.exists() or VOICE.stat().st_size==0:
        raise RuntimeError("TTS did not create a usable audio file.")
    print("🔊",VOICE)
else:
    print("⚠️ No narration requested.")

## 6 — رندر الفيديو وعرضه

In [ ]:
#@title 🎥 Render MP4
from IPython.display import Video,display
if RUN_RENDER:
    cmd=[sys.executable,"-u","-m","voice2reel.production_pipeline",
         "--plan",str(PLAN_PATH),"--lock",str(LOCK),"--output",str(VIDEO)]
    if RUN_TTS:
        if not VOICE.is_file() or VOICE.stat().st_size==0:
            raise FileNotFoundError("Run voiceover cell before rendering.")
        cmd+=["--voice",str(VOICE)]
    if MUSIC.is_file():cmd+=["--music",str(MUSIC)]
    p=subprocess.run(cmd,cwd=str(REPO),text=True,stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT,timeout=3600)
    print(p.stdout,flush=True)
    if p.returncode:raise RuntimeError("Rendering failed; see log above.")
    if not VIDEO.is_file() or VIDEO.stat().st_size==0:
        raise RuntimeError("Missing or empty video.")
    display(Video(str(VIDEO),embed=False,width=320))
    print("🎬",VIDEO,"MB:",round(VIDEO.stat().st_size/1e6,2))
else:
    print("⏭ Rendering disabled")